# DoubleIntegrator — Stage 3 — Evaluation & Cross-Method Comparison

Evaluate the configured controllers on matched scenarios.


## 1. Set up the system and load the stage config

Load dependencies and configuration.


In [ ]:
"""Boilerplate: make the in-repo `sdpc` package importable and resolve this system."""
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "sdpc").is_dir() and (candidate / "case_studies").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the SD-DPC repository")

REPO_ROOT = find_repo_root()
CASE_DIR = REPO_ROOT / "case_studies" / "DoubleIntegrator"
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
from sdpc.config import load_config, seed_everything
from sdpc.registry import make_system

SYSTEM = "double_integrator"
device = torch.device("cpu")
system = make_system(SYSTEM, device=device)
CONFIGS = CASE_DIR / "configs"
RESULTS = CASE_DIR / "results"

print(f"System        : {SYSTEM}")
print(f"State dim nx  : {system.nx}")
print(f"Control dim nu: {system.nu}")
print(f"Sample time ts: {system.ts}")
print(f"Input bounds  : [{system.umin}, {system.umax}]")
print(f"State bounds  : [{system.xmin}, {system.xmax}]")
print(f"Discrete model: {system.is_discrete}")


In [ ]:
cfg = load_config(CONFIGS / 'eval.yaml')
seed_everything(int(cfg.get('seed', 0)))
print('Methods being compared:', cfg.get('methods'))
print('Number of seeds       :', cfg['n_seeds'])
print('Evaluation steps      :', cfg['eval_nsteps'])
print('SD-DPC/MPC horizon    :', cfg['nsteps'])
print('Shared DPC weights    :', cfg['weights'])

## 2. Load the trained SINDy model and sparse policy

Restore the required checkpoints.


In [ ]:
from sdpc.io import (
    find_dynamics_checkpoint,
    find_nn_checkpoint,
    find_policy_checkpoint,
)
from sdpc.sindy import load_model
from sdpc.training import load_nn_policy

dynamics_path = find_dynamics_checkpoint(RESULTS, cfg)
policy_path = find_policy_checkpoint(RESULTS, cfg)
nn_path = find_nn_checkpoint(RESULTS, cfg)
sindy = load_model(dynamics_path, device=device)
policy = load_model(policy_path, device=device)
nn_policy = load_nn_policy(system, nn_path, cfg, device=device)
print('Dynamics :', dynamics_path)
print('SD-DPC   :', policy_path)
print('NN-DPC   :', nn_path)
print('NN loss match:', getattr(nn_policy, 'dpc_loss_reference_verified', None))

## 3. Run the comparison across seeds

Compare the configured methods.


In [ ]:
from sdpc.eval import dpc_loss_reference, evaluate

OUT_DIR = RESULTS / 'eval' / 'controller_comparison'
print('Canonical loss reference:', dpc_loss_reference(system, cfg))
out = evaluate(
    system, sindy, policy, cfg, device,
    out_dir=OUT_DIR, nn_policy=nn_policy,
)
print(f"Evaluated {len(cfg.get('methods', []))} methods over {cfg['n_seeds']} seeds.")
print('Saved metrics and trajectories to', OUT_DIR)


## 4. Aggregated metric table

Compare the configured methods.


In [ ]:
from sdpc.plotting import format_metrics_table

print(format_metrics_table(out['summary']))

In [ ]:
import pandas as pd
df = pd.DataFrame.from_dict(out['summary'], orient='index')
df.columns

In [ ]:
df[['final_tracking_mse_mean',
       'final_tracking_mse_std']]

## 5. Per-seed detail for one metric

Compare the configured methods.


In [ ]:
import numpy as np

metric = 'tracking_mse'
for method, rows in out['per_seed'].items():
    vals = [r[metric] for r in rows if metric in r]
    print(f'{method:>24}: ' + ', '.join(f'{v:.3g}' for v in vals))

## 6. Inspect one validation trajectory

Select any saved scenario and compare all controllers.


In [ ]:
import matplotlib.pyplot as plt
from sdpc.plotting import plot_trajectories_with_obstacles

TRAJECTORY_INDEX = 1  # 0, ..., cfg['n_seeds'] - 1
if not 0 <= TRAJECTORY_INDEX < len(out['scenarios']):
    raise IndexError(f'TRAJECTORY_INDEX must be in [0, {len(out["scenarios"]) - 1}]')

scenario = out['scenarios'][TRAJECTORY_INDEX]
styles = {
    'mpc': ('MPC', '#5A9367'),
    'sd_dpc': ('SD-DPC', '#4472C4'),
    'nn_dpc': ('NN-DPC', '#F4A261'),
}
trajectories = [
    {
        'traj': out['trajectories'][method]['x_traj'][TRAJECTORY_INDEX:TRAJECTORY_INDEX + 1],
        'label': label, 'color': color, 'linewidth': 1.5, 'show_start': False,
    }
    for method, (label, color) in styles.items()
]

fig, ax = plot_trajectories_with_obstacles(
    trajectories, [cfg['ellipse']], scenario['r'],
    obstacle_band=0.0, xmin=system.xmin, xmax=system.xmax,
    state_bounds=(system.xmin, system.xmax), show_initial=True,
    publication=True, title='',
)
print('trajectory:', TRAJECTORY_INDEX, 'seed:', out['per_seed']['mpc'][TRAJECTORY_INDEX]['seed'])
print('initial state:', scenario['xn'][0, 0].tolist())
print('reference:', scenario['r'][0, -1].tolist())
fig.savefig(OUT_DIR / 'trajectory.pdf')
plt.show()
